# Caduceus: Bi-Directional Equivariant Long-Range DNA Sequence Modeling
**ArXivist-generated reproduction notebook**
Paper: https://arxiv.org/abs/2403.03234 (Schiff et al., ICML 2024)
Generated: 2026-07-24

This notebook (1) verifies the paper's central architectural claim — **RC equivariance**
(Theorem 3.1) — on CPU, then (2) loads the **official pretrained Caduceus** from HuggingFace and
fine-tunes it on a **Genomic Benchmarks** task.

**Runtime:** the architecture demo/tests run on CPU. Fine-tuning the official weights needs a
**GPU** (Runtime > Change runtime type > T4 GPU) because Caduceus uses the Mamba CUDA kernels.

## 1. Environment check

In [ ]:
import sys, torch
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 2. Get the code

In [ ]:
# If running in Colab, clone your fork (this repo lives under paper-repos/arxiv_2403_003234).
import os
if not os.path.exists("src/caduceus"):
    # Adjust to your fork URL, or upload the arxiv_2403_003234 folder and cd into it.
    # !git clone https://github.com/<you>/arxivist.git
    # %cd arxivist/workspace/paper-repos/arxiv_2403_003234
    pass
print("cwd:", os.getcwd())
print("has src/caduceus:", os.path.isdir("src/caduceus"))

## 3. Install dependencies

In [ ]:
import subprocess
r = subprocess.run(["pip","install","-q","-e","."], capture_output=True, text=True)
print((r.stdout or r.stderr)[-600:])
# GPU-only Mamba kernels for the OFFICIAL forward pass (skip on CPU; sections 4-5 still run):
if torch.cuda.is_available():
    subprocess.run(["pip","install","-q","mamba-ssm","causal-conv1d"], text=True)
    print("installed mamba-ssm / causal-conv1d")

## 4. Paper overview

**Problem.** DNA modeling needs (a) bi-directional context, (b) reverse-complement (RC) symmetry —
either DNA strand carries the same info — and (c) long range (regulatory effects up to 1 Mbp away).

**Key ideas.**
- **BiMamba** — bi-directional Mamba with *weight tying*: `y = M(x) + flip(M(flip(x)))`, ~1x params.
- **MambaDNA** — RC equivariance via channel split: `concat(M(x_a), RC(M(RC(x_b))))`, shared `M`.
  Provably `RC∘M = M∘RC` (**Theorem 3.1**).
- **Caduceus-PS** (parameter sharing, RC-equivariant embed + LM head) and **Caduceus-Ph**
  (post-hoc conjoining: average `f(x)` and `f(RC(x))` at inference).

Pretrained with MLM on the human genome; beats HyenaDNA/Mamba and, on long-range VEP, 10x larger models.

## 5. Verify RC equivariance (Theorem 3.1) — CPU, no kernels

The paper's core claim is that MambaDNA is *exactly* RC-equivariant. We check it on the pure-PyTorch
reference (`src/caduceus/models/rc_equivariance.py`). The gap should be ~0.

In [ ]:
import torch
from src.caduceus.models.rc_equivariance import (
    BiMamba, MambaDNA, RCEquivariantEmbedding, reverse_complement_tensor)

torch.manual_seed(0)

# BiMamba preserves shape
bi = BiMamba(dim=16)
x = torch.randn(2, 32, 16)
print("BiMamba:", tuple(x.shape), "->", tuple(bi(x).shape))

# MambaDNA satisfies RC(MambaDNA(x)) == MambaDNA(RC(x))
m = MambaDNA(dim=16).eval()
xi = torch.randn(4, 24, 16)
with torch.no_grad():
    lhs = reverse_complement_tensor(m(xi))
    rhs = m(reverse_complement_tensor(xi))
gap = (lhs - rhs).abs().max().item()
print(f"MambaDNA output mean-abs: {m(xi).abs().mean().item():.4f}")
print(f"RC-equivariance gap (max abs): {gap:.2e}   <-- Theorem 3.1 holds if ~0")
assert gap < 1e-4, "equivariance broken!"
print("Theorem 3.1 VERIFIED (exact equivariance).")

Run the full test suite (7 checks: equivariance, composition, embedding, shapes):

In [ ]:
import subprocess
r = subprocess.run(["python","-m","pytest","tests/","-q"], capture_output=True, text=True)
print(r.stdout[-800:])
print(r.stderr[-300:] if r.returncode else "all tests passed")

## 6. Load the official pretrained Caduceus (GPU)

Loads `kuleshov-group/caduceus-ph_seqlen-1k_d_model-118_n_layer-4_lr-8e-3` via
`AutoModel(trust_remote_code=True)`. Needs the Mamba kernels (GPU).

In [ ]:
from src.caduceus.models.classifier import CaduceusClassifier
from src.caduceus.data.tokenizer import CharTokenizer

MODEL = "kuleshov-group/caduceus-ph_seqlen-1k_d_model-118_n_layer-4_lr-8e-3"
tok = CharTokenizer(MODEL, max_len=500)
print("tokenizer vocab:", tok.vocab_size)

try:
    model = CaduceusClassifier.from_pretrained(
        MODEL, num_classes=2, variant="ph", device=str(device),
        rc_complement_ids=tok.complement_id_map())
    print(model)
    print("params:", round(sum(p.numel() for p in model.parameters())/1e6, 2), "M")
except Exception as e:
    print("Load failed (need GPU + mamba-ssm):", str(e)[:300])

## 7. Download a Genomic Benchmarks task (via API)

In [ ]:
import subprocess
r = subprocess.run(["python","data/download.py","--benchmark","genomic_benchmarks",
                    "--task","human_nontata_promoters"], capture_output=True, text=True)
print((r.stdout or r.stderr)[-500:])

## 8. Fine-tune (Caduceus-Ph)

Mean-pool + linear head, RC data augmentation, post-hoc conjoining at eval. Paper target for
`human_nontata_promoters` (Caduceus-Ph): **0.946 accuracy**.

In [ ]:
import subprocess
r = subprocess.run(["python","train.py","--config","configs/config.yaml"],
                   capture_output=True, text=True)
print(r.stdout[-2000:])
print(r.stderr[-500:] if r.returncode else "")

Faster sanity check on the easiest task (`demo_human_or_worm`, paper 0.973):

In [ ]:
# subprocess.run(["python","data/download.py","--task","demo_human_or_worm"])
# subprocess.run(["python","train.py","--config","configs/config.yaml",
#                 "--task","demo_human_or_worm"])

## 9. Paper results for comparison (Table 1, Caduceus-Ph)

In [ ]:
paper = {
    "human_nontata_promoters": 0.946,
    "demo_human_or_worm": 0.973,
    "human_enhancers_ensembl": 0.893,
    "human_ocr_ensembl": 0.828,
    "demo_coding_vs_intergenomic_seqs": 0.915,
    "dummy_mouse_enhancers_ensembl": 0.754,
}
print("Caduceus-Ph reported top-1 accuracy (paper Table 1, 5-fold CV):")
for k, v in paper.items():
    print(f"  {k:36s} {v}")
print("\nPaste your fine-tuned accuracy back to ArXivist for the Stage 6 comparison report.")

## What to do next

1. **Verify architecture:** section 5 (works on CPU — Theorem 3.1 checked exactly).
2. **Fine-tune (GPU):** section 8; try other tasks with `--task`.
3. **Evaluate a checkpoint:** `python evaluate.py --config configs/config.yaml --checkpoint checkpoints/best.pt`
4. **Compare:** paste your accuracy back to ArXivist (Stage 6).

**Recipe (Sec 5.2.1):** AdamW, cosine, mean-pool head, RC aug + post-hoc conjoining, LR ∈ {1e-3, 2e-3}, 5-fold CV.